# 18xA — Development-frozen trading strategy selection

This stage constructs a simple long-only trading rule from development data only. For each complete eleven-contract book, it selects the contract with the largest raw edge \(p^{model}-p^{market}\) and buys one YES share only when that edge exceeds a threshold. The observed pre-cutoff YES price is a fill proxy, not a bid or ask quote. Thresholds are selected at a hypothetical one-cent all-in cost per share, with cost and staleness sensitivity retained for later analysis.

**Revision v2.** Maximum drawdown is measured relative to a running peak that includes the initial portfolio value of zero. Strategy selection and every non-drawdown result are unchanged.

In [1]:
from __future__ import annotations
import hashlib, json, math, platform, sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
ROOT=Path.cwd().resolve()
if not (ROOT/'.git').exists(): raise RuntimeError(f'Run from repository root, not {ROOT}')
UTC=timezone.utc

def sha(path:Path)->str:
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def parse_bool(s:pd.Series,name:str)->pd.Series:
    if pd.api.types.is_bool_dtype(s): return s.astype(bool)
    out=s.astype(str).str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False,'yes':True,'no':False})
    if out.isna().any(): raise ValueError(f'Cannot parse Boolean {name}: {s[out.isna()].drop_duplicates().tolist()}')
    return out.astype(bool)

def verify_manifest(path:Path):
    m=pd.read_csv(path); failures=[]
    for r in m.itertuples(index=False):
        p=ROOT/r.path
        if not p.is_file(): failures.append(f'MISSING {r.path}'); continue
        if sha(p)!=r.sha256: failures.append(f'HASH {r.path}')
        if p.stat().st_size!=int(r.size_bytes): failures.append(f'SIZE {r.path}')
    if failures: raise AssertionError(f'Manifest failed {path}:\\n'+'\\n'.join(failures))

def write_manifest(out:Path, report_dir:Path, filename:str):
    rows=[]
    for root in [out,report_dir]:
        for p in sorted(root.rglob('*')):
            if p.is_file() and p.name!=filename:
                rows.append({'path':str(p.relative_to(ROOT)),'size_bytes':p.stat().st_size,'sha256':sha(p)})
    pd.DataFrame(rows).to_csv(out/filename,index=False)

def save_frame(frame:pd.DataFrame,path:Path):
    x=frame.copy()
    for c in x.columns:
        if pd.api.types.is_datetime64_any_dtype(x[c]):
            if getattr(x[c].dt,'tz',None) is not None: x[c]=x[c].astype('string')
            else: x[c]=x[c].dt.strftime('%Y-%m-%d')
    x.to_csv(path,index=False)

def max_drawdown(daily:pd.Series)->float:
    if daily.empty: return float('nan')
    cumulative=daily.sort_index().cumsum().to_numpy(dtype=float)
    running_peak=np.maximum.accumulate(np.concatenate(([0.0],cumulative)))[1:]
    return float(np.min(cumulative-running_peak))

STEP='18xA'; PRIMARY_COST=0.01; STALENESS_CAP=3.0
THRESHOLDS=np.array([0.00,0.02,0.04,0.06,0.08,0.10,0.15,0.20])
COSTS=np.array([0.00,0.005,0.01,0.02,0.05])
MAN=ROOT/'data/manual/18x_outcome_free_market_prices'
WA=ROOT/'data/processed/18wA_contract_probability_mapping'; WB=ROOT/'data/processed/18wB_development_probability_calibration'; VC=ROOT/'data/processed/18vC_common_support_scoring_and_selection'
MARKET=MAN/'18x_outcome_free_market_price_panel.csv'; META=MAN/'18x_outcome_free_market_price_metadata.json'
UNCAL=WA/'18wA_development_uncalibrated_probability_panel.csv'; WA_SUM=WA/'18wA_summary.json'; WA_MAN=WA/'18wA_sha256_manifest.csv'
CAL=WB/'18wB_development_calibrated_probability_panel.csv'; WB_SUM=WB/'18wB_summary.json'; WB_MAN=WB/'18wB_sha256_manifest.csv'
SELECT=VC/'18vC_selected_candidates.json'; VC_SUM=VC/'18vC_summary.json'; VC_MAN=VC/'18vC_sha256_manifest.csv'
OUT=ROOT/'data/processed/18xA_development_trading_selection'; REPORT=ROOT/'reports/18xA_development_trading_selection'; OUT.mkdir(parents=True,exist_ok=True); REPORT.mkdir(parents=True,exist_ok=True)
for p in [MARKET,META,UNCAL,WA_SUM,WA_MAN,CAL,WB_SUM,WB_MAN,SELECT,VC_SUM,VC_MAN]:
    if not p.is_file(): raise FileNotFoundError(p)
for p in [WA_MAN,WB_MAN,VC_MAN]: verify_manifest(p)
for name,p in [('18wA',WA_SUM),('18wB',WB_SUM),('18vC',VC_SUM)]:
    if json.loads(p.read_text()).get('verdict')!='PASS': raise AssertionError(f'{name} not PASS')
meta=json.loads(META.read_text())
if meta.get('output_sha256')!=sha(MARKET) or meta.get('forbidden_outcome_columns_present') is not False: raise AssertionError('Outcome-free market metadata failed')
market=pd.read_csv(MARKET,dtype={'market_id':str},low_memory=False); uncal=pd.read_csv(UNCAL,dtype={'market_id':str},low_memory=False); cal=pd.read_csv(CAL,dtype={'market_id':str},low_memory=False)
for d in [market,uncal,cal]: d['event_date']=pd.to_datetime(d.event_date,errors='raise')
forbidden={'hko_daily_max_c','Y_event_int','Y_no_int','market_binary_brier','market_binary_log_score'}
if forbidden.intersection(market.columns): raise AssertionError('Outcome-free market panel contains outcomes')
with SELECT.open() as f: selection=json.load(f)
primary_candidate=selection['selected_overall']
uncal=uncal.copy(); uncal['probability_variant']='UNCALIBRATED'; uncal['p_model']=uncal.p_model_uncalibrated
cal=cal.copy(); cal['probability_variant']='CALIBRATED'; cal['p_model']=cal.p_model_calibrated
prob=pd.concat([uncal,cal],ignore_index=True)
if len(prob)!=8976 or prob.event_date.max()>pd.Timestamp('2026-05-21'): raise AssertionError('Unexpected development probability panel')
key=['candidate_id','probability_variant','event_date','decision_rule']
mcols=['event_date','market_id','decision_rule','p_market','decision_cutoff_utc','selected_price_timestamp_utc','price_staleness_hours']
opp=prob.merge(market[mcols],on=['event_date','market_id','decision_rule'],how='inner',validate='many_to_one')
counts=opp.groupby(key).size(); complete=counts[counts.eq(11)].reset_index()[key]
opp=opp.merge(complete.assign(complete_market_book=True),on=key,how='inner')
stale=opp.groupby(key).price_staleness_hours.max().rename('book_max_staleness_hours').reset_index(); opp=opp.merge(stale,on=key,validate='many_to_one')
opp=opp.loc[opp.book_max_staleness_hours.le(STALENESS_CAP)].copy(); opp['edge']=opp.p_model-opp.p_market
if len(opp)!=6996: raise AssertionError(f'Expected 6996 development opportunity rows, found {len(opp)}')
best=opp.sort_values(key+['edge','p_model','p_market','market_id'],ascending=[True,True,True,True,False,False,True,True],kind='mergesort').groupby(key,as_index=False).head(1).reset_index(drop=True)
if len(best)!=636: raise AssertionError(f'Expected 636 best-edge books, found {len(best)}')
perf=[]
for keys,g in best.groupby(['candidate_id','probability_variant','decision_rule'],sort=True):
    for threshold in THRESHOLDS:
        trade=g.loc[g.edge.ge(threshold)]
        for cost in COSTS:
            pnl=trade.Y_event_int-trade.p_market-cost; capital=trade.p_market+cost
            daily=pd.Series(pnl.to_numpy(),index=trade.event_date).groupby(level=0).sum() if len(trade) else pd.Series(dtype=float)
            perf.append({'candidate_id':keys[0],'probability_variant':keys[1],'decision_rule':keys[2],'threshold':float(threshold),'cost_per_share':float(cost),'opportunity_books':len(g),'opportunity_dates':g.event_date.nunique(),'trade_count':len(trade),'trade_dates':trade.event_date.nunique(),'trade_rate':len(trade)/len(g),'total_net_pnl':float(pnl.sum()),'mean_net_pnl_per_trade':float(pnl.mean()) if len(trade) else np.nan,'total_capital':float(capital.sum()),'return_on_capital':float(pnl.sum()/capital.sum()) if capital.sum()>0 else np.nan,'hit_rate':float((pnl>0).mean()) if len(trade) else np.nan,'mean_selected_edge':float(trade.edge.mean()) if len(trade) else np.nan,'max_drawdown':max_drawdown(daily)})
performance=pd.DataFrame(perf)
if len(performance)!=960: raise AssertionError(f'Expected 960 threshold rows, found {len(performance)}')
primary=performance.loc[performance.cost_per_share.eq(PRIMARY_COST)&performance.trade_count.ge(8)&performance.trade_dates.ge(8)].copy()
cost2=performance.loc[performance.cost_per_share.eq(0.02),['candidate_id','probability_variant','decision_rule','threshold','total_net_pnl']].rename(columns={'total_net_pnl':'total_net_pnl_cost_0_02'})
primary=primary.merge(cost2,on=['candidate_id','probability_variant','decision_rule','threshold'],validate='one_to_one')
selected=[]
for candidate_id,g in primary.groupby('candidate_id'):
    row=g.sort_values(['total_net_pnl','total_net_pnl_cost_0_02','mean_net_pnl_per_trade','threshold','probability_variant','decision_rule'],ascending=[False,False,False,False,True,True],kind='mergesort').iloc[0].to_dict(); selected.append(row)
registry=pd.DataFrame(selected)
family=prob[['candidate_id','model_family']].drop_duplicates(); registry=registry.merge(family,on='candidate_id',validate='one_to_one')
registry['strategy_role']=np.where(registry.candidate_id.eq(primary_candidate),'PRIMARY_OVERALL',np.where(registry.model_family.eq('GAUSSIAN_PROCESS'),'GAUSSIAN_PROCESS_FAMILY','TREE_FAMILY'))
registry['primary_cost_per_share']=PRIMARY_COST; registry['market_staleness_cap_hours']=STALENESS_CAP; registry['position_size_shares']=1.0; registry['trade_side']='BUY_YES_ONLY'; registry['entry_price_interpretation']='OBSERVED_PRE_CUTOFF_YES_PRICE_FILL_PROXY'
expected={
'pooled_empirical_residual':('CALIBRATED','event_day_open',0.08,7.12),
'gp_matern32_rule':('CALIBRATED','24h_prior',0.08,4.1825),
'catboost_quantile_pooled':('CALIBRATED','12h_prior',0.04,4.958),}
for cid,(variant,rule,threshold,pnl) in expected.items():
    r=registry.loc[registry.candidate_id.eq(cid)]
    if len(r)!=1 or r.probability_variant.iloc[0]!=variant or r.decision_rule.iloc[0]!=rule or not np.isclose(r.threshold.iloc[0],threshold) or not np.isclose(r.total_net_pnl.iloc[0],pnl): raise AssertionError(f'Unexpected selected strategy for {cid}: {r.to_dict("records")}')
selected_books=[]
for r in registry.itertuples(index=False):
    g=best.loc[best.candidate_id.eq(r.candidate_id)&best.probability_variant.eq(r.probability_variant)&best.decision_rule.eq(r.decision_rule)].copy(); g['strategy_role']=r.strategy_role; g['selected_threshold']=r.threshold; g['trade_flag']=g.edge.ge(r.threshold); g['primary_cost_per_share']=PRIMARY_COST; g['net_pnl_primary_cost']=np.where(g.trade_flag,g.Y_event_int-g.p_market-PRIMARY_COST,0.0); selected_books.append(g)
selected_books=pd.concat(selected_books,ignore_index=True)
if len(selected_books)!=77 or int(selected_books.trade_flag.sum())!=71: raise AssertionError('Selected development book counts differ')
checks=pd.DataFrame([
{'check':'outcome_free_market_source','passed':not bool(forbidden.intersection(market.columns)),'detail':'market source excludes outcomes and scores','blocking':True},
{'check':'development_probability_rows_8976','passed':len(prob)==8976,'detail':str(len(prob)),'blocking':True},
{'check':'opportunity_rows_6996','passed':len(opp)==6996,'detail':str(len(opp)),'blocking':True},
{'check':'best_edge_books_636','passed':len(best)==636,'detail':str(len(best)),'blocking':True},
{'check':'threshold_rows_960','passed':len(performance)==960,'detail':str(len(performance)),'blocking':True},
{'check':'selected_strategies_3','passed':len(registry)==3,'detail':str(len(registry)),'blocking':True},
{'check':'development_only','passed':best.event_date.max()<=pd.Timestamp('2026-05-21'),'detail':'no holdout or June outcome used','blocking':True},
{'check':'raw_market_prices_not_normalised','passed':True,'detail':'edge uses raw p_market','blocking':True},
{'check':'drawdown_includes_zero_initial_value','passed':performance.max_drawdown.le(1e-12).all(),'detail':'running peak includes initial portfolio value zero','blocking':True},
])
if not checks.passed.all(): raise AssertionError(checks.loc[~checks.passed].to_string(index=False))
issues=pd.DataFrame(columns=['issue_level','issue_code','candidate_id','event_date','decision_rule','detail','blocking'])
frames={'development_trade_opportunity_panel':opp,'development_best_edge_book_panel':best,'development_threshold_performance':performance,'selected_strategy_registry':registry,'selected_development_book_panel':selected_books,'integrity_checks':checks,'issues':issues}
for name,frame in frames.items(): save_frame(frame,OUT/f'18xA_{name}.csv')
protocol={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','strategy':'single highest positive model-minus-market edge per complete date-rule book; buy one YES share if edge reaches threshold','market_price_interpretation':'observed selected pre-cutoff YES price used as a fill proxy, not a bid/ask quote','threshold_grid':THRESHOLDS.tolist(),'cost_grid':COSTS.tolist(),'primary_cost_per_share':PRIMARY_COST,'market_staleness_cap_hours':STALENESS_CAP,'minimum_development_trades':8,'minimum_development_trade_dates':8,'selection':'for each pre-selected family candidate maximise development total net PnL at primary cost, then cost-0.02 PnL, mean PnL, higher threshold','model_selection_reopened':False,'holdout_or_external_outcomes_loaded':False,'market_prices_normalised':False,'short_or_no_trades_allowed':False,'maximum_drawdown_definition':'minimum cumulative PnL relative to running peak including initial portfolio value zero'}
(OUT/'18xA_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
summary={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','development_probability_rows':8976,'development_opportunity_contract_rows':6996,'development_best_edge_books':636,'threshold_performance_rows':960,'selected_strategies':3,'selected_development_books':77,'selected_development_trades':71,'selected_strategies_detail':registry[['strategy_role','candidate_id','probability_variant','decision_rule','threshold','total_net_pnl']].to_dict('records'),'holdout_or_external_outcomes_loaded':False,'issue_rows':0,'integrity_checks_passed':int(checks.passed.sum()),'integrity_checks_total':len(checks)}
(OUT/'18xA_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
sources=pd.DataFrame([{'input_role':'outcome_free_market_price_panel','path':str(MARKET.relative_to(ROOT)),'rows':len(market),'sha256':sha(MARKET)},{'input_role':'outcome_free_market_price_metadata','path':str(META.relative_to(ROOT)),'rows':1,'sha256':sha(META)},{'input_role':'18wA_development_uncalibrated_probability_panel','path':str(UNCAL.relative_to(ROOT)),'rows':len(uncal),'sha256':sha(UNCAL)},{'input_role':'18wB_development_calibrated_probability_panel','path':str(CAL.relative_to(ROOT)),'rows':len(cal),'sha256':sha(CAL)},{'input_role':'18vC_selected_candidates','path':str(SELECT.relative_to(ROOT)),'rows':1,'sha256':sha(SELECT)}]); sources.to_csv(OUT/'18xA_source_inventory.csv',index=False)
(OUT/'18xA_environment.json').write_text(json.dumps({'generated_at_utc':datetime.now(UTC).isoformat(),'python':sys.version,'platform':platform.platform(),'pandas':pd.__version__,'numpy':np.__version__,'revision':'v2'},indent=2),encoding='utf-8')
lines=['# 18xA development-frozen trading strategy selection','','**PASS**','','The observed pre-cutoff YES price is a fill proxy rather than a contemporaneous bid or ask. The strategy buys one YES share in the highest-edge contract of a complete eleven-contract book.','','| Role | Candidate | Variant | Rule | Threshold | Development net PnL at 0.01 cost |','|---|---|---|---|---:|---:|']
for r in registry.sort_values('strategy_role').itertuples(index=False): lines.append(f'| {r.strategy_role} | {r.candidate_id} | {r.probability_variant} | {r.decision_rule} | {r.threshold:.2f} | {r.total_net_pnl:.4f} |')
(REPORT/'18xA_development_trading_selection_report.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
write_manifest(OUT,REPORT,'18xA_sha256_manifest.csv')
print(json.dumps(summary,indent=2)); print('18xA PASS')

{
  "step": "18xA",
  "generated_at_utc": "2026-07-22T15:21:45.732228+00:00",
  "verdict": "PASS",
  "development_probability_rows": 8976,
  "development_opportunity_contract_rows": 6996,
  "development_best_edge_books": 636,
  "threshold_performance_rows": 960,
  "selected_strategies": 3,
  "selected_development_books": 77,
  "selected_development_trades": 71,
  "selected_strategies_detail": [
    {
      "strategy_role": "TREE_FAMILY",
      "candidate_id": "catboost_quantile_pooled",
      "probability_variant": "CALIBRATED",
      "decision_rule": "12h_prior",
      "threshold": 0.04,
      "total_net_pnl": 4.958
    },
    {
      "strategy_role": "GAUSSIAN_PROCESS_FAMILY",
      "candidate_id": "gp_matern32_rule",
      "probability_variant": "CALIBRATED",
      "decision_rule": "24h_prior",
      "threshold": 0.08,
      "total_net_pnl": 4.1825
    },
    {
      "strategy_role": "PRIMARY_OVERALL",
      "candidate_id": "pooled_empirical_residual",
      "probability_variant": "